# Notebook 1: Data Collection

Pulls mention counts for each presidential candidate in the 12 months before their election day.

**Primary source:** GDELT Global Knowledge Graph (GKG) — free, covers ~800 news sources globally, available from 1979 onward via BigQuery.

**For 1960–1979:** ProQuest Historical Newspapers via manual or API query (NYT, WaPo, Chicago Tribune). Documented separately.

See `PREREGISTRATION.md` for the full data source rationale and `data/edge_cases.md` for search term decisions.

In [ ]:
import pandas as pd
import requests
import time
from datetime import datetime, timedelta
from pathlib import Path

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(exist_ok=True)

## Election schedule

Defines candidates, election dates, and search terms for each year.

In [ ]:
# Each entry: (year, election_date, [(candidate_name, search_term, party), ...])
# Search terms follow decisions in data/edge_cases.md

ELECTIONS = [
    (1960, '1960-11-08', [('Kennedy', 'Kennedy', 'D'), ('Nixon', 'Nixon', 'R')]),
    (1964, '1964-11-03', [('Johnson', 'Johnson', 'D'), ('Goldwater', 'Goldwater', 'R')]),
    (1968, '1968-11-05', [('Nixon', 'Nixon', 'R'), ('Humphrey', 'Humphrey', 'D')]),
    (1972, '1972-11-07', [('Nixon', 'Nixon', 'R'), ('McGovern', 'McGovern', 'D')]),
    (1976, '1976-11-02', [('Carter', 'Carter', 'D'), ('Ford', 'Ford', 'R')]),
    (1980, '1980-11-04', [('Reagan', 'Reagan', 'R'), ('Carter', 'Carter', 'D')]),
    (1984, '1984-11-06', [('Reagan', 'Reagan', 'R'), ('Mondale', 'Mondale', 'D')]),
    (1988, '1988-11-08', [('Bush', 'Bush', 'R'), ('Dukakis', 'Dukakis', 'D')]),
    (1992, '1992-11-03', [('Clinton', 'Clinton', 'D'), ('Bush', 'Bush', 'R'), ('Perot', 'Perot', 'I')]),
    (1996, '1996-11-05', [('Clinton', 'Clinton', 'D'), ('Dole', 'Dole', 'R')]),
    (2000, '2000-11-07', [('Bush', 'Bush', 'R'), ('Gore', 'Gore', 'D')]),
    (2004, '2004-11-02', [('Bush', 'Bush', 'R'), ('Kerry', 'Kerry', 'D')]),
    (2008, '2008-11-04', [('Obama', 'Obama', 'D'), ('McCain', 'McCain', 'R')]),
    (2012, '2012-11-06', [('Obama', 'Obama', 'D'), ('Romney', 'Romney', 'R')]),
    (2016, '2016-11-08', [('Trump', 'Trump', 'R'), ('Clinton', 'Clinton', 'D')]),
    (2020, '2020-11-03', [('Biden', 'Biden', 'D'), ('Trump', 'Trump', 'R')]),
    (2024, '2024-11-05', [('Trump', 'Trump', 'R'), ('Harris', 'Harris', 'D')]),
]

## GDELT GKG Query (1979–2024)

GDELT's free API (`gdeltdoc` Python package) allows full-text search of ~800 news sources.
Returns article count per time window. We query 12 months ending on Election Day.

**Install:** `pip install gdeltdoc`

In [ ]:
from gdeltdoc import GdeltDoc, Filters

gd = GdeltDoc()

def query_gdelt_mentions(search_term, start_date, end_date):
    """
    Returns article count for search_term between start_date and end_date.
    Dates as 'YYYY-MM-DD' strings.
    """
    f = Filters(
        keyword=search_term,
        start_date=start_date,
        end_date=end_date,
        country='US',
    )
    articles = gd.article_search(f)
    return len(articles) if articles is not None else 0


def get_window(election_date_str):
    election_date = datetime.strptime(election_date_str, '%Y-%m-%d')
    start = election_date - timedelta(days=365)
    return start.strftime('%Y-%m-%d'), election_date_str

In [ ]:
# Run queries for all GDELT-covered elections (1979+)
# WARNING: GDELT free API rate-limits; add sleep between calls

results = []

for year, election_date, candidates in ELECTIONS:
    if year < 1979:
        print(f'{year}: Skipping — pre-GDELT era, use ProQuest (see notebook section below)')
        continue

    start, end = get_window(election_date)
    print(f'\n{year}: querying {start} → {end}')

    for name, term, party in candidates:
        count = query_gdelt_mentions(term, start, end)
        print(f'  {name}: {count:,}')
        results.append({
            'year': year,
            'candidate': name,
            'party': party,
            'search_term': term,
            'window_start': start,
            'window_end': end,
            'source': 'GDELT',
            'raw_mentions': count,
        })
        time.sleep(2)  # respect rate limits

df_raw = pd.DataFrame(results)
df_raw.to_csv(RAW_DIR / 'gdelt_mentions.csv', index=False)
print('\nSaved to data/raw/gdelt_mentions.csv')
df_raw.head(20)

## Pre-GDELT Era: 1960–1979

GDELT does not reliably cover news before 1979. For the four elections in this range
(1960, 1964, 1968, 1972, 1976), we use the **Google Books Ngram Viewer API** as a proxy
for print media saturation. This is a rougher signal but the only freely available
systematic source for this era.

Alternative (requires institutional access): ProQuest Historical Newspapers fulltext search.
If you have access, use the ProQuest API and document your query strings in `data/raw/proquest_queries.md`.

**Limitation acknowledged in pre-registration:** Source heterogeneity between eras is
a known confound. Sensitivity analysis will test whether results hold for 1979–2024 only.

In [ ]:
import urllib.request
import json

def query_ngrams(word, year_start, year_end, corpus='en-2019', smoothing=0):
    """
    Google Books Ngram Viewer API.
    Returns average frequency per million words for the given word in the year range.
    """
    url = (
        f'https://books.google.com/ngrams/json?content={word}'
        f'&year_start={year_start}&year_end={year_end}'
        f'&corpus={corpus}&smoothing={smoothing}'
    )
    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read())
    if not data:
        return 0.0
    # Return sum of frequencies over the window
    return sum(data[0]['timeseries'])


early_results = []

for year, election_date, candidates in ELECTIONS:
    if year >= 1979:
        continue
    window_start_year = year - 1
    print(f'{year}: querying Google Ngrams {window_start_year}–{year}')
    for name, term, party in candidates:
        freq = query_ngrams(term, window_start_year, year)
        print(f'  {name}: {freq:.6f}')
        early_results.append({
            'year': year,
            'candidate': name,
            'party': party,
            'search_term': term,
            'window_start': str(window_start_year),
            'window_end': str(year),
            'source': 'Google_Ngrams',
            'raw_mentions': freq,
        })
        time.sleep(1)

df_early = pd.DataFrame(early_results)
df_early.to_csv(RAW_DIR / 'ngrams_mentions.csv', index=False)
print('Saved to data/raw/ngrams_mentions.csv')
df_early